# PSY197B — Data Exploration & Sanity Checks

Quick notebook for verifying data shapes, label distributions, and sample plots  
before running the full ML pipeline.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

PROJECT_ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

from data.loader import load_config, load_dataset, find_latest_run

print(f"Project root: {PROJECT_ROOT}")

## 1. Load Configuration

In [ ]:
config = load_config(os.path.join(PROJECT_ROOT, 'configs', 'config.yaml'))

print('Subjects:', config['data']['subjects'])
print('Conditions:', config['data']['conditions_available'])
print('Epoch window:', config['data']['epoch_window_ms'], 'ms')
print('CLIP categories:', config.get('clip_categories', []))

## 2. Check Raw Data Availability

In [ ]:
data_root = os.path.join(PROJECT_ROOT, config['data']['root'])
print(f'Data root: {data_root}\n')

for sj in config['data']['subjects']:
    sj_dir = os.path.join(data_root, f'sj{sj:02d}')
    print(f'--- Subject {sj:02d} ---')
    for subdir in ['eeg', 'beh', 'eye']:
        path = os.path.join(sj_dir, subdir)
        if os.path.isdir(path):
            files = os.listdir(path)
            print(f'  {subdir}/: {len(files)} items')
        else:
            print(f'  {subdir}/: MISSING')

## 3. Check Pipeline Run Outputs

In [ ]:
try:
    latest = find_latest_run(PROJECT_ROOT)
    print(f'Latest run: {latest}')
    run_data = os.path.join(PROJECT_ROOT, 'runs', latest, 'data')
    if os.path.isdir(run_data):
        files = sorted(os.listdir(run_data))
        for f in files:
            size_mb = os.path.getsize(os.path.join(run_data, f)) / 1e6
            print(f'  {f:50s} {size_mb:8.2f} MB')
except FileNotFoundError as e:
    print(f'No runs found: {e}')

## 4. Load Dataset & Check Label Distribution

In [ ]:
dataset = load_dataset(config, PROJECT_ROOT)

print(f'\nTotal no-go trials: {dataset.total_trials}')
print(f'Channels: {dataset.n_channels}')
print(f'Timepoints: {dataset.n_timepoints}')
print(f'Subjects: {dataset.subject_ids}')
print(f'Loaded conditions: {dataset.conditions_loaded}')
print(f'Missing conditions: {dataset.conditions_missing}')

In [ ]:
for sd in dataset.subject_data:
    print(f'\nsj{sd.subject_id:02d} / {sd.condition}:')
    print(f'  EEG shape: {sd.eeg.shape}')
    print(f'  Labels: CR={sd.n_correct_rejection}, FA={sd.n_false_alarm}')
    print(f'  Fixation data: {"yes" if sd.fixation_sequences else "no"}')
    if sd.fixation_sequences:
        lens = [len(s) for s in sd.fixation_sequences]
        print(f'  Fixation seq lengths: mean={np.mean(lens):.1f}, '
              f'min={min(lens)}, max={max(lens)}')

## 5. Sample EEG Epoch Plots

In [ ]:
if dataset.subject_data:
    sd = dataset.subject_data[0]
    sfreq = config['data']['sfreq']
    tmin = config['eeg']['tmin']
    tmax = config['eeg']['tmax']
    times = np.linspace(tmin * 1000, tmax * 1000, sd.eeg.shape[2])

    cr_idx = np.where(sd.labels == 1)[0]
    fa_idx = np.where(sd.labels == 0)[0]

    ch_idx = 0  # first channel
    fig, ax = plt.subplots(figsize=(10, 4))

    if len(cr_idx) > 0:
        cr_mean = sd.eeg[cr_idx, ch_idx, :].mean(axis=0) * 1e6
        ax.plot(times, cr_mean, label=f'Correct Rejection (n={len(cr_idx)})',
                color='#3498db')
    if len(fa_idx) > 0:
        fa_mean = sd.eeg[fa_idx, ch_idx, :].mean(axis=0) * 1e6
        ax.plot(times, fa_mean, label=f'False Alarm (n={len(fa_idx)})',
                color='#e74c3c')

    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'sj{sd.subject_id:02d} / {sd.condition} — Channel 0')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Class Balance Summary

In [ ]:
balance_data = []
for sd in dataset.subject_data:
    balance_data.append({
        'Subject': f'sj{sd.subject_id:02d}',
        'Condition': sd.condition,
        'Correct Rejections': sd.n_correct_rejection,
        'False Alarms': sd.n_false_alarm,
        'Total': len(sd.labels),
        'CR Rate': sd.n_correct_rejection / max(len(sd.labels), 1),
    })

balance_df = pd.DataFrame(balance_data)
print(balance_df.to_string(index=False))

if len(balance_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    x = range(len(balance_df))
    ax.bar(x, balance_df['Correct Rejections'], label='CR', color='#3498db')
    ax.bar(x, balance_df['False Alarms'],
           bottom=balance_df['Correct Rejections'],
           label='FA', color='#e74c3c')
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{r['Subject']}\n{r['Condition']}" for _, r in balance_df.iterrows()],
        fontsize=8
    )
    ax.set_ylabel('Trial Count')
    ax.set_title('No-Go Trial Class Balance')
    ax.legend()
    plt.tight_layout()
    plt.show()